# 07 · RAG — Contextual Chunking: protocolo de actuacion (Fase C-bis)

Este cuaderno documenta, con codigo real y salidas reales contra la base de datos,
el protocolo completo seguido para anadir **una frase de contexto por chunk**
al corpus del RAG (tecnica de *contextual retrieval* de Anthropic) usando un
modelo externo (Gemini, a traves de Antigravity) como colaborador humano-en-el-bucle,
sin exponerlo nunca a la base de datos ni al trabajo ya hecho por Ollama.

**Por que existe este cuaderno**: no es solo un registro de lo que se hizo, es un
**protocolo reusable** para la proxima vez que un proyecto necesite delegar una
tarea de generacion masiva a un LLM externo bajo un presupuesto de sesion limitado,
con verificacion automatica y sin poder revisar el resultado a mano.

**Fuentes:**
- `methodology/ROADMAP_RAG_Y_GRAFO.md` (seccion 3, "Contextual chunking") y
  `methodology/ROADMAP_RAG_Y_GRAFO_v2.md` (estado de la pieza).
- [Contextual Retrieval in AI Systems — Anthropic](https://www.anthropic.com/engineering/contextual-retrieval)
- `src/README.md` (entradas `rag_export_context_batches.py`,
  `rag_import_context_batches.py`, `rag_export_context_repairs.py`,
  `rag_reembed_context.py`, `rag_query_compare.py`).
- `data/rag_context_batches/README.md` (ciclo operativo dia a dia, gitignored,
  vive solo en local).


In [1]:
import os
import re
import inspect
from pathlib import Path

import pandas as pd
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent

load_dotenv(REPO_ROOT / ".env")

def connect():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=os.getenv("POSTGRES_PORT", "5432"),
        dbname=os.environ["POSTGRES_DB"],
        user=os.environ["POSTGRES_USER"],
        password=os.environ["POSTGRES_PASSWORD"],
    )

def query(sql, params=None):
    with connect() as conn:
        return pd.read_sql(sql, conn, params=params)

print("Conexion configurada. REPO_ROOT =", REPO_ROOT)


Conexion configurada. REPO_ROOT = C:\Users\rubbe\Documents\ALCheMy\.claude\worktrees\rag-contextual-chunking-notebook


## 1. Por que existe esta pieza — lo que dice el roadmap

La tecnica no es una idea propia: es *contextual retrieval*, publicada por Anthropic,
y estaba citada en el roadmap del proyecto desde la version 1, marcada como
**pendiente de implementar**. La cita textual (`ROADMAP_RAG_Y_GRAFO.md`):

> *"Contextual chunking: antes de vectorizar, un LLM barato anade 1-2 frases de
> contexto a cada chunk [...]. Es la tecnica de Anthropic — 35% menos fallos por
> si sola."*

Y el resultado que reporta Anthropic al combinarla con busqueda lexica (BM25) y
un reranker: **-67% de fallos de recuperacion** (de 5,7% a 1,9%). Solo los
*contextual embeddings*, sin las otras dos piezas, ya dan **-35%**.

El detalle que importa para todo lo que sigue: la tecnica dice **"antes de
vectorizar"**. El contexto no se guarda al lado del vector como una etiqueta
bonita — **se funde con el texto del chunk y ESO es lo que se vectoriza**. Si
no se hace asi, el trabajo entero de generar las frases no mejora la busqueda
en absoluto: el motor de busqueda solo compara vectores, nunca lee el texto
guardado al lado.


## 2. Arquitectura: aislar al colaborador externo del trabajo ya validado

Tres piezas, cada una con una responsabilidad y un limite de escritura muy
concreto:

```mermaid
flowchart TD
    DOCS[("rag_documents<br/>1 fila por PDF/paper, catálogo bibliográfico")]
    CHUNKS["<b>rag_chunks</b> — 1 fila por fragmento de texto<br/>• text ← splitter recursivo determinista, SIN LLM<br/>• embedding ← vector de Ollama (bge-m3) del chunk SOLO — Fase B, ya cerrada<br/>• embedding_v2 ← vector de Ollama del (contexto + chunk) — Fase D, columna nueva, NUNCA sobrescribe embedding"]
    CTX["<b>rag_chunk_context</b> — tabla sidecar: SOLO la frase de contexto que escribe Gemini/Antigravity<br/>• context_sentence &nbsp; • source_model &nbsp; • batch_file<br/>Ningún script de esta fase escribe nunca en rag_chunks.text ni en rag_chunks.embedding"]

    DOCS --> CHUNKS
    CHUNKS -->|"chunk_id (FK)"| CTX

    classDef prod fill:#E3F2FD,stroke:#1E88E5,color:#0D47A1
    classDef side fill:#FFF3E0,stroke:#FB8C00,color:#E65100
    class DOCS,CHUNKS prod
    class CTX side
```

**Por que una tabla sidecar y no una columna en `rag_chunks` directamente:**
un modelo externo, con un historial ya conocido de errores (ver seccion 5),
nunca debe tener un camino de escritura que toque una columna de la que ya
depende produccion (`embedding`). Si el sidecar sale mal, se borra y se
rehace sin haber arriesgado nada. Este mismo principio es el que despues
justifica `embedding_v2` en la seccion 7: ni siquiera el paso final de
vectorizar el contexto se permite sobrescribir el original.


In [2]:
df = query('''
    SELECT
        (SELECT count(*) FROM rag_documents WHERE sensitivity = 'PUBLIC' AND needs_ocr = FALSE) AS documentos_public,
        (SELECT count(*) FROM rag_chunks) AS chunks_totales,
        (SELECT count(*) FROM rag_chunk_context) AS frases_de_contexto,
        (SELECT count(*) FROM rag_chunks WHERE embedding_v2 IS NOT NULL) AS chunks_con_embedding_v2;
''')
df


C:\Users\rubbe\AppData\Local\Temp\ipykernel_30044\2989295823.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,documentos_public,chunks_totales,frases_de_contexto,chunks_con_embedding_v2
0,956,87765,87765,87765


## 3. Paso 1 — exportar lotes por documento, nunca por chunk

`src/rag_export_context_batches.py` escribe un JSON por documento en
`data/rag_context_batches/pending/`. La razon de agrupar por documento (no por
chunk) es la misma razon de ser de la tecnica: el modelo necesita ver el
**documento completo** para poder situar cada fragmento dentro de el — un
chunk aislado no tiene nada contra que "situarse". Agrupar por documento
tambien reduce ~87.765 peticiones a nivel de chunk a ~956 a nivel de documento.

El filtro de privacidad es una clausula `WHERE` en la consulta SQL, no una
instruccion de prompt que el modelo podria ignorar:


In [3]:
src = (REPO_ROOT / "src" / "rag_export_context_batches.py").read_text(encoding="utf-8")
start = src.index("PENDING_QUERY")
triple = chr(34) * 3
print(src[start:start + src.index(triple, start + 20) + 3])


PENDING_QUERY = """
SELECT DISTINCT d.document_id, d.file_name, d.file_path, d.title, d.authors,
       d.year, d.journal, d.doi
FROM rag_documents d
JOIN rag_chunks c ON c.document_id = d.document_id
LEFT JOIN rag_chunk_context cc ON cc.chunk_id = c.id
WHERE d.sensitivity = 'PUBLIC'
  AND d.needs_ocr = FALSE
  AND cc.chunk_id IS NULL
ORDER BY d.document_id;
"""

CHUNKS_QUERY = """
SELECT id, text FROM rag_chunks WHERE document_id = %s ORDER BY id;
"""


def slugify(name):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name)[:60]


def main():
    parser = argparse.ArgumentParser(description="Export per-document context batches")
    parser.add_argument("--limit", type=int, default=None,
                        help="export at most N documents (for a pilot run)")
    parser.add_argument("--doc", type=int, default=None,
                        help="export only this one document_id")
    args = parser.parse_args()

    PENDING_DIR.mkdir(parents=True, exist_ok=Tru

## 4. El prompt para Gemini / Antigravity

Este es el prompt tal cual se pego en el chat de Antigravity al principio de
cada sesion, apuntando a la carpeta `pending/`. Incluye ya el refuerzo anadido
tras el primer incidente (seccion 5.1): la instruccion explicita de no
reciclar frases.


> You will find JSON files in data/rag_context_batches/pending/ (or
> data/rag_context_batches/repairs/pending/ for a repair batch). Each one
> lists chunk_ids and the full text of the document they belong to. Follow
> each file's own "instructions" field exactly — one specific,
> non-generic sentence per chunk\_id, grounded in what that exact chunk
> says. Never a generic template, never a "(part N)" counter standing in
> for a real difference, never a garbled or truncated word filling a gap,
> and never the same sentence \(or a near-identical one\) reused across
> multiple chunks. Process the files one at a time, in filename order,
> moving each finished one to the matching done/ folder as you go. Stop
> cleanly when pending/ is empty or you run out of session budget. Same
> hard limits as always: never edit any .py file, never touch the
> database, never open anything outside data/rag_context_batches/.

Version **autonoma** (Antigravity con acceso a terminal, ejecuta ella misma
el ciclo completo en bucle hasta agotar presupuesto o terminar el corpus —
ver seccion 6):

> You also have permission to run terminal commands from the repository
> root. Instead of stopping after one wave, run the full cycle in a loop:
> 1. Run `python src/rag_export_context_batches.py --limit 30`. If it
> exports 0 new files, the corpus is done — stop and report it. 2. Process
> every file now in pending/, exactly as above, moving each to done/ as
> you finish it. 3. Once pending/ is empty, run
> `python src/rag_import_context_batches.py --apply`. 4. Run
> `python src/rag_import_context_batches.py --status` and note the
> numbers. 5. Repeat from step 1. Stop when: you run out of session
> budget \(finish the file you are on first\), OR step 1 has nothing left
> to export, OR a command errors. Hard limits: only ever run those two
> exact commands, never raw SQL, never a database client, never edit any
> .py file. If `--apply` reports rejected files, leave them in rejected/
> untouched and just count them in your summary.


## 5. Verificar ANTES de escribir — nunca confiar en que el formato correcto implique contenido correcto

`src/rag_import_context_batches.py` nunca escribe nada en `rag_chunk_context`
sin pasar primero por `verify()`. Si un archivo falla, se rechaza **entero**
— nunca una importacion parcial de un documento — y se mueve a `rejected/`
para que el fallo sea ruidoso, no un hueco silencioso descubierto semanas
despues.


In [4]:
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
import rag_import_context_batches as importer

print(inspect.getsource(importer.verify))


def verify(batch, file_name):
    """Returns (ok, reason). Never raises - a malformed file is a rejection,
    not a crash."""
    expected_ids = {c["chunk_id"] for c in batch.get("chunks", [])}
    if not expected_ids:
        return False, "el batch original no tenia chunks"

    results = batch.get("results")
    if not isinstance(results, list):
        return False, "falta la clave 'results' o no es una lista"

    if len(results) != len(expected_ids):
        return False, "se esperaban {} sentencias, llegaron {}".format(
            len(expected_ids), len(results))

    got_ids = []
    for item in results:
        if not isinstance(item, dict) or "chunk_id" not in item or "context_sentence" not in item:
            return False, "un elemento de 'results' no tiene chunk_id/context_sentence"
        if not isinstance(item["context_sentence"], str) or not item["context_sentence"].strip():
            return False, "context_sentence vacia para chunk_id={}".format(item.get("chunk_id

### 5.1 Incidente 1 — frases recicladas (el primer lote real)

El primer lote grande procesado por Gemini paso la verificacion estructural
(mismo numero de `chunk_id`, sin huecos) pero, al inspeccionar el contenido,
reutilizaba **3 a 10 frases genericas** para documentos de 100-250 chunks —
por ejemplo, 227 chunks con solo 3 frases distintas repetidas en bucle. Nada
en las reglas 1-4 de `verify()` lo detectaba porque son puramente
estructurales.

**Fix**: regla 5 anadida a `verify()` — si un archivo tiene mas de 5
resultados y menos del 50% de las frases son unicas entre si, se rechaza.
Probado retroactivamente contra los 30 archivos del incidente: **30/30
detectados**, 0 falsos positivos contra archivos ya buenos.

### 5.2 Incidente 2 — la plantilla rota `"(part N)"`

Un fallo mas sutil, encontrado *despues* de que la regla 5 ya estuviera en
produccion: para una minoria de chunks por documento, el modelo rellenaba una
plantilla rigida — `"This X chunk (part N) evaluates <fragmento roto>
regarding <titulo>"` — con una palabra truncada o sin sentido en el hueco.
Cada frase es **tecnicamente distinta** (cambia el numero de parte y el
fragmento), asi que pasa la regla de diversidad sin problema. Solo se
descubre escaneando el texto ya importado en busca del patron.


In [5]:
df = query(r'''
    SELECT count(*) FILTER (WHERE context_sentence ~ '\(part \d+\)') AS frases_con_plantilla_rota,
           count(*) AS total_frases
    FROM rag_chunk_context;
''')
df["porcentaje"] = (100 * df["frases_con_plantilla_rota"] / df["total_frases"]).round(3)
df


C:\Users\rubbe\AppData\Local\Temp\ipykernel_30044\2989295823.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,frases_con_plantilla_rota,total_frases,porcentaje
0,0,87765,0.0


### 5.3 Incidente 3 — el mismo fallo del 5.1, pero anterior al fix

Al auditar el corpus entero (no solo los lotes recientes) aparecieron **37
documentos** con el fallo original de frases recicladas — pero su
`created_at` era **anterior** a que la regla 5 existiera, asi que nunca
pasaron por ese filtro. No es que el filtro fallara: es una franja de datos
mas antigua que quedo fuera de su alcance por construccion.

**Leccion**: un chequeo anadido a un pipeline solo protege lo que se procesa
*despues* de anadirlo. Los datos ya escritos antes necesitan una auditoria
retroactiva aparte — nunca asumir que "ya lo arregle" cubre el pasado.


In [6]:
df = query('''
    SELECT count(*) FILTER (WHERE ratio < 0.5) AS documentos_bajo_diversidad,
           count(*) AS documentos_evaluados
    FROM (
        SELECT c.document_id,
               count(DISTINCT cc.context_sentence)::float / count(*) AS ratio
        FROM rag_chunk_context cc
        JOIN rag_chunks c ON c.id = cc.chunk_id
        GROUP BY c.document_id
        HAVING count(*) > 5
    ) sub;
''')
df


C:\Users\rubbe\AppData\Local\Temp\ipykernel_30044\2989295823.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,documentos_bajo_diversidad,documentos_evaluados
0,0,953


## 6. Reparar quirurgicamente, no rehacer el corpus entero

`src/rag_export_context_repairs.py` tiene dos modos, uno por cada tipo de
incidente, porque su alcance dentro de un documento es distinto:

- **`--mode template`** (incidente 5.2): dentro de un documento afectado, solo
  una minoria de chunks estaba mal (mediana <10%). Reexportar el documento
  entero tiraria a la basura cientos de frases buenas para arreglar unas
  pocas. Se exportan **solo los `chunk_id` marcados**, con el texto completo
  del documento como contexto.
- **`--mode recycled`** (incidente 5.3): aqui la mayoria de los chunks de un
  documento comparten frase, y no hay forma fiable de saber cual copia era
  "la real" para un chunk dado. Se reexporta el **documento completo**.

Ambos importan con el mismo `verify()` de siempre, apuntando a un arbol de
carpetas separado (`data/rag_context_batches/repairs/`) para que el trabajo
de reparacion nunca se mezcle con el procesamiento de documentos nuevos:


In [7]:
print("python src/rag_export_context_repairs.py --mode template")
print("python src/rag_export_context_repairs.py --mode recycled")
print("python src/rag_import_context_batches.py --batch-root data/rag_context_batches/repairs --apply")


python src/rag_export_context_repairs.py --mode template
python src/rag_export_context_repairs.py --mode recycled
python src/rag_import_context_batches.py --batch-root data/rag_context_batches/repairs --apply


## 7. Trabajar por oleadas con presupuesto de sesion limitado

Antigravity (la suscripcion de Google AI Pro que ejecuta Gemini) tiene un
limite de horas por sesion. El protocolo que hace esto seguro de interrumpir
en cualquier momento:

1. `--status` antes de cada oleada: si `pending/` no esta vacio, hay trabajo
   a medias de la sesion anterior — se retoma esa misma carpeta, nunca se
   exporta una oleada nueva encima.
2. `rag_export_context_batches.py --limit N` exporta *solo* los documentos
   que aun no tienen ninguna frase — repetir el comando nunca duplica
   trabajo.
3. Si la sesion de Antigravity se corta a medias, lo hecho ya esta a salvo en
   `done/`; lo que falta sigue en `pending/` esperando a la siguiente sesion.
4. `--apply` solo escribe lo que paso `verify()`; el resto va a `rejected/`
   sin tocar la base de datos.

Estado final verificado del corpus tras cerrar los tres incidentes:


In [8]:
df = query('''
    SELECT
        (SELECT count(*) FROM rag_documents WHERE sensitivity = 'PUBLIC' AND needs_ocr = FALSE) AS documentos_totales,
        (SELECT count(DISTINCT d.document_id)
           FROM rag_documents d JOIN rag_chunks c ON c.document_id = d.document_id
           JOIN rag_chunk_context cc ON cc.chunk_id = c.id
          WHERE d.sensitivity = 'PUBLIC') AS documentos_con_frase,
        (SELECT count(*) FROM rag_chunks c JOIN rag_documents d ON d.document_id = c.document_id
          WHERE d.sensitivity = 'PUBLIC' AND d.needs_ocr = FALSE) AS chunks_totales,
        (SELECT count(*) FROM rag_chunk_context) AS chunks_con_frase;
''')
df["porcentaje_corpus"] = (100 * df["chunks_con_frase"] / df["chunks_totales"]).round(1)
df


C:\Users\rubbe\AppData\Local\Temp\ipykernel_30044\2989295823.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,documentos_totales,documentos_con_frase,chunks_totales,chunks_con_frase,porcentaje_corpus
0,956,956,87765,87765,100.0


## 8. Fase D — de "frase guardada" a "frase que influye en la busqueda"

Terminar la tabla `rag_chunk_context` **no mejora ninguna busqueda por si
solo** — es solo texto guardado. Para que la tecnica de Anthropic haga lo que
promete hace falta el paso que el propio nombre de la tecnica describe:
vectorizar `contexto + chunk` junto, no el chunk solo.

**Decision de diseno explicita**: esto se escribe en una columna **nueva**
(`rag_chunks.embedding_v2`), nunca sobrescribiendo `embedding`. La razon es
directa — las frases de contexto las escribio un modelo externo con un
historial de errores ya documentado en este mismo cuaderno (secciones 5.1 a
5.3), y no es razonable pedirle al dueno del proyecto que revise a mano
~88.000 frases para confiar en ellas. Con una columna aparte, el peor caso
posible es `UPDATE rag_chunks SET embedding_v2 = NULL` — cero riesgo para la
busqueda que ya funciona hoy.

`src/rag_reembed_context.py` calcula `embedding_v2` en local con el mismo
modelo Ollama (`bge-m3`) que ya se uso para `embedding`, mismo patron de
lotes y reintentos que `rag_ingest_chunks.py`:


In [9]:
src = (REPO_ROOT / "src" / "rag_reembed_context.py").read_text(encoding="utf-8")
start = src.index("query = ")
end = src.index("cursor.execute(query, params)") + len("cursor.execute(query, params)")
print(src[start:end])


query = """
        SELECT c.id, c.text, cc.context_sentence
        FROM rag_chunks c
        JOIN rag_chunk_context cc ON cc.chunk_id = c.id
        WHERE {}
        ORDER BY c.id
    """.format(" AND ".join(where))
    if args.limit is not None:
        query += " LIMIT {}".format(int(args.limit))

    cursor.execute(query, params)


In [10]:
df = query('''
    SELECT count(*) FILTER (WHERE embedding_v2 IS NOT NULL) AS ya_vectorizados,
           count(*) AS total_chunks
    FROM rag_chunks;
''')
df["porcentaje"] = (100 * df["ya_vectorizados"] / df["total_chunks"]).round(2)
df


C:\Users\rubbe\AppData\Local\Temp\ipykernel_30044\2989295823.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ya_vectorizados,total_chunks,porcentaje
0,87765,87765,100.0


### 8.1 Comparacion A/B — juzgar sin revisar frase a frase

`src/rag_query_compare.py` hace una busqueda por similitud de coseno pura
(sin fusion BM25) contra `embedding` y contra `embedding_v2` para la misma
pregunta, y las imprime lado a lado. Es la forma de decidir si el contexto
ayuda de verdad, con preguntas reales, sin depender de leer las 88.000
frases generadas.

Salida real capturada durante el desarrollo de esta fase (piloto de 100
chunks del paper de Salcedo-Abraira et al. 2023 sobre MOFs de Cu para
fotocatalisis de agua):

```
Pregunta: photocatalytic water splitting MOF
==============================================================================

--- SIN contexto (embedding, lo que usa rag_query.py hoy) ---

  1. similitud 0.7753   Wei Wang et al. (2017) DOI: 10.1002/advs.201600371
     limited timescales, ranging from several hours to a few days. Robust
     MOF-derived materials that can operate on a longer timescale are thus
     needed to go a step further toward the real application of water
     splitting. 3.2. Photocatalytic Water Splitting Some MOFs have shown
     favorable photocatalytic activity for water split...

--- CON contexto (embedding_v2) ---

  1. similitud 0.6321   Pablo Salcedo-Abraira et al. (2023) DOI: 10.1021/acs.chemmater.3c00054
     Two Cu-Based Phosphonate Metal-Organic Frameworks as Efficient Water-
     Splitting Photocatalysts...

Chunks que aparecen SOLO con contexto: 3
Chunks que aparecen SOLO sin contexto : 3
Chunks en ambos                       : 0
```

Con solo 100 chunks vectorizados (de un unico documento) la comparacion
todavia no es representativa del corpus — este es el resultado de la prueba
piloto que confirmo que el mecanismo funciona de punta a punta, no una
evaluacion de calidad. La evaluacion real se hace cuando `embedding_v2` este
completo (seccion 8, tabla de progreso) lanzando varias preguntas reales del
dominio y comparando que pasajes trae cada columna.

**Importante**: las similitudes de coseno de `embedding` y `embedding_v2` no
son directamente comparables en valor absoluto (anadir la frase de contexto
desplaza el vector en el espacio de bge-m3) — lo que hay que juzgar es si los
**pasajes** que trae cada version son mas relevantes para la pregunta, no si
el numero de similitud es mayor o menor.


## 9. Protocolo generalizable — checklist para el proximo proyecto

Lo que queda de esta fase, mas alla del RAG de ALCheMy, es un patron
reusable para cualquier tarea futura que delegue trabajo masivo a un LLM
externo:

1. **Filtro de privacidad como clausula SQL, nunca como instruccion de
   prompt.** Un modelo puede ignorar una instruccion; no puede ignorar una
   fila que la consulta nunca le entrego.
2. **Tabla sidecar, no columna nueva en la tabla de produccion**, mientras el
   colaborador externo este generando el dato. Aisla el peor caso.
3. **Verificar ANTES de escribir, y rechazar la unidad entera** (nunca una
   escritura parcial) si algo no cuadra — un fallo ruidoso es barato de
   arreglar; un hueco silencioso se descubre semanas despues.
4. **Un chequeo estructural (formato correcto) no es un chequeo de
   contenido.** El incidente 5.1 lo demostro: un archivo perfectamente
   formado puede llevar basura dentro. Hacen falta las dos capas.
5. **Un chequeo anadido al pipeline solo protege lo que se procesa despues.**
   Auditar retroactivamente lo que ya paso por el pipeline antes del fix es
   un paso aparte, no gratis (incidente 5.3).
6. **Reparar quirurgicamente cuando el dano esta disperso; rehacer la unidad
   completa cuando el dano es mayoritario dentro de ella.** No hay una regla
   unica — depende de si se puede identificar con confianza que parte
   concreta esta mal.
7. **Lotes por la unidad que necesita el contexto completo** (aqui,
   documento), no por la unidad mas pequena (chunk) — aunque eso multiplique
   las llamadas.
8. **Handoff por archivos + un comando de estado** cuando la herramienta
   externa tiene un presupuesto de sesion limitado. Nunca depender de que
   una sesion larga no se corte.
9. **Cuando el resultado final alimenta un vector ya en produccion, escribe
   en una columna nueva y compara A/B con preguntas reales** antes de decidir
   si sustituye a la version actual. Revisar a mano miles de items generados
   por IA no es una opcion realista — comparar resultados de busqueda si lo
   es.
